In [ ]:
!pip uninstall -y peft accelerate transformers -q
!pip install -q \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  scikit-learn \
  sentencepiece

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR   = "/content/drive/MyDrive/CLIP_ViT_L14_MirageNews"
CHECKPOINT = "/content/drive/MyDrive/CLIP_ViT_L14_MirageNews/checkpoint-1875"

import os
import torch
import numpy as np
import functools

from datasets import load_dataset
from transformers import (
    CLIPProcessor,
    CLIPModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from torch import nn
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ─── Dataset ─────────────────────────────────────────────────────────────────
dataset   = load_dataset("anson-huang/mirage-news")
model_name = "openai/clip-vit-large-patch14"
processor  = CLIPProcessor.from_pretrained(model_name)
clip_model = CLIPModel.from_pretrained(model_name).to(device)

def preprocess(example):
    inputs = processor(
        text=example["text"],
        images=example["image"],
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )
    return {
        "input_ids":      inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values":   inputs["pixel_values"][0],
        "labels":         example["label"]
    }

encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names
)

# ─── Model ───────────────────────────────────────────────────────────────────
class CLIPForMFND(nn.Module):
    def __init__(self, clip_model, num_labels=2):
        super().__init__()
        self.clip       = clip_model
        hidden          = clip_model.config.projection_dim
        self.classifier = nn.Linear(hidden * 2, num_labels)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )
        fused  = torch.cat([outputs.text_embeds, outputs.image_embeds], dim=1)
        logits = self.classifier(fused)
        loss   = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return {"loss": loss, "logits": logits}

model = CLIPForMFND(clip_model).to(device)

# ─── Metrics ─────────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision,
        "recall":    recall,
        "f1":        f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

# ─── Training args ───────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
# ─── Trainer ─────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# ─── Fix PyTorch 2.6 torch.load issue ────────────────────────────────────────
_original_torch_load = torch.load
torch.load = functools.partial(_original_torch_load, weights_only=False)

# ─── Resume từ checkpoint ─────────────────────────────────────────────────────
trainer.train(resume_from_checkpoint=CHECKPOINT)

trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("✅ Model saved to:", SAVE_DIR)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
4,0.094800,0.088632,0.993600,0.991242,0.996000,0.993615,0.999753
5,0.062100,0.085013,0.989600,0.987261,0.992000,0.989625,0.998875


✅ Model saved to: /content/drive/MyDrive/CLIP_ViT_L14_MirageNews


In [ ]:
trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

In [ ]:
# ─── Test Evaluation — tất cả 5 test sets ────────────────────────────────────
test_splits = [
    "test1_nyt_mj",
    "test2_bbc_dalle",
    "test3_cnn_dalle",
    "test4_bbc_sdxl",
    "test5_cnn_sdxl"
]

for split in test_splits:
    results = trainer.predict(encoded_dataset[split])
    logits  = results.predictions
    labels  = results.label_ids
    probs   = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds   = np.argmax(probs, axis=1)

    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    print(f"\n{'='*50}")
    print(f"  {split}")
    print(f"{'='*50}")
    print(f"  Accuracy : {accuracy_score(labels, preds)*100:.2f}%")
    print(f"  Precision: {p*100:.2f}%")
    print(f"  Recall   : {r*100:.2f}%")
    print(f"  F1       : {f1*100:.2f}%")
    print(f"  AUC      : {roc_auc_score(labels, probs[:, 1]):.5f}")
    print(f"{'='*50}")


  test1_nyt_mj
  Accuracy : 100.00%
  Precision: 100.00%
  Recall   : 100.00%
  F1       : 100.00%
  AUC      : 1.00000



  test2_bbc_dalle
  Accuracy : 49.00%
  Precision: 47.37%
  Recall   : 18.00%
  F1       : 26.09%
  AUC      : 0.75593



  test3_cnn_dalle
  Accuracy : 55.20%
  Precision: 75.00%
  Recall   : 15.60%
  F1       : 25.83%
  AUC      : 0.89041



  test4_bbc_sdxl
  Accuracy : 67.00%
  Precision: 72.73%
  Recall   : 54.40%
  F1       : 62.24%
  AUC      : 0.85386



  test5_cnn_sdxl
  Accuracy : 82.60%
  Precision: 91.37%
  Recall   : 72.00%
  F1       : 80.54%
  AUC      : 0.96161


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
for f in os.listdir("/content/drive/MyDrive/CLIP_ViT_L14_MirageNews"):
    print(f)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoint-2500
checkpoint-3125
model.safetensors
training_args.bin
preprocessor_config.json
tokenizer_config.json
special_tokens_map.json
vocab.json
merges.txt
tokenizer.json


In [ ]:
import os, torch, time, numpy as np
from torch import nn
from transformers import CLIPProcessor, CLIPModel
from safetensors.torch import load_file
from PIL import Image

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

N_WARMUP = 10
N_RUNS   = 100

checkpoint_dir = "/content/drive/MyDrive/CLIP_ViT_L14_MirageNews"
model_name     = "openai/clip-vit-large-patch14"

# ─── Model class ─────────────────────────────────────────────────────────────
class CLIPForMFND(nn.Module):
    def __init__(self, clip_model, num_labels=2):
        super().__init__()
        self.clip       = clip_model
        hidden          = clip_model.config.projection_dim
        self.classifier = nn.Linear(hidden * 2, num_labels)

    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values
        )
        fused  = torch.cat([outputs.text_embeds, outputs.image_embeds], dim=1)
        logits = self.classifier(fused)
        loss   = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}

# ─── Load model ───────────────────────────────────────────────────────────────
print("⏳ Loading CLIP ViT-L/14...")
clip_model = CLIPModel.from_pretrained(model_name)
model      = CLIPForMFND(clip_model)
state      = load_file(f"{checkpoint_dir}/model.safetensors")
model.load_state_dict(state, strict=False)
model      = model.to(device)
model.eval()

processor = CLIPProcessor.from_pretrained(checkpoint_dir)
print("✅ Model loaded")

# ─── Dummy input ─────────────────────────────────────────────────────────────
dummy_image = Image.new("RGB", (224, 224), color=(128, 128, 128))
dummy_text  = "This is a sample news headline for inference measurement."
inputs = processor(
    text=dummy_text, images=dummy_image,
    truncation=True, padding="max_length", max_length=77, return_tensors="pt"
)
input_ids      = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)
pixel_values   = inputs["pixel_values"].to(device)

# ─── Đo ──────────────────────────────────────────────────────────────────────
try:
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    with torch.no_grad():
        for _ in range(N_WARMUP):
            model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)

    latencies = []
    with torch.no_grad():
        for _ in range(N_RUNS):
            if device == "cuda": torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values)
            if device == "cuda": torch.cuda.synchronize()
            latencies.append((time.perf_counter() - t0) * 1000)

    latencies = np.array(latencies)

    if device == "cuda":
        vram = torch.cuda.max_memory_allocated() / 1024**2
    else:
        import psutil
        vram = psutil.Process(os.getpid()).memory_info().rss / 1024**2

    print(f"\n{'='*50}")
    print(f"📊 CLIP ViT-L/14")
    print(f"{'='*50}")
    print(f"  Latency — mean: {latencies.mean():.2f} ms | median: {np.median(latencies):.2f} ms | std: {latencies.std():.2f} ms")
    mem_label = "VRAM" if device == "cuda" else "RAM"
    print(f"  {mem_label} (peak allocated): {vram:.1f} MB")

except Exception as e:
    print(f"❌ Lỗi: {e}")
finally:
    model.cpu()
    if device == "cuda": torch.cuda.empty_cache()

Using device: cuda
⏳ Loading CLIP ViT-L/14...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["bos_token_id"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["eos_token_id"]` will be overriden.


✅ Model loaded

📊 CLIP ViT-L/14
  Latency — mean: 84.14 ms | median: 84.26 ms | std: 1.52 ms
  VRAM (peak allocated): 6558.9 MB
